# Thí nghiệm NHANES với 15 chỉ số — CNN 1D và Logistic Regression

**Dữ liệu là NHANES của CDC Hoa Kỳ, không phải WHO hoặc người Việt Nam.** Trang microdata WHO STEPS Việt Nam chưa tải được trong môi trường hiện tại. Notebook huấn luyện độc lập và không thay đổi mô hình website.

Mục tiêu: phân loại câu trả lời DIQ010 về việc **từng được nhân viên y tế thông báo mắc đái tháo đường**, không dự báo mắc mới hay chẩn đoán. Glucose/HbA1c có thể liên quan đến chẩn đoán đã có; tình trạng điều trị làm thay đổi chỉ số. Các biến được đo đồng thời với nhãn. Không dùng mô hình cho sàng lọc hoặc con người thật. Chỉ xem đây là thực nghiệm học thuật.

Nguồn và mã biến: [NHANES 2015–2016](https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?BeginYear=2015), [2017–2018](https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?BeginYear=2017), [DIQ010](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DIQ_J.htm). Các file XPT gốc và SHA-256 được lưu trong manifest; báo cáo kèm giới hạn. Dữ liệu thuộc phân mẫu nhịn đói, kết quả không dùng trọng số khảo sát nên không đại diện dân số. Không khẳng định hai chu kỳ là kiểm định ngoài lâm sàng.

## Cách sử dụng
Chạy tuần tự từ trên xuống bằng **Run All**. Mỗi phần có mục đích riêng; các biến giữa các cell phụ thuộc nhau. Hai chu kỳ là NHANES Hoa Kỳ, không phải WHO hay dữ liệu Việt Nam. Chạy lại sẽ ghi đè các tệp dẫn xuất và artifact trong các đường dẫn bên dưới; bản manifest snapshot được cập nhật riêng theo hướng dẫn trong `data/metadata/NHANES15.md`.

## 1. Thiết lập môi trường
### 1.1. Nạp thư viện
Các thư viện phục vụ xử lý dữ liệu, xây dựng CNN và mô hình so sánh.

In [ ]:
from pathlib import Path
import json, hashlib, random, sys, platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score, accuracy_score, confusion_matrix, brier_score_loss)
import torch
from torch import nn
import joblib

### 1.2. Đường dẫn và seed
Tìm thư mục dự án, chỉ định nơi đọc dữ liệu gốc và lưu kết quả, cố định seed để thuận tiện tái lập.

In [ ]:
ROOT = Path.cwd().resolve()
if not (ROOT / 'docker-compose.yml').exists():
    ROOT = next((p for p in ROOT.parents if (p / 'docker-compose.yml').exists()), None)
if ROOT is None:
    raise RuntimeError('Mở notebook từ bên trong thư mục dự án.')
RAW = ROOT / 'data/raw/nhanes'
DERIVED = ROOT / 'data/raw/nhanes15_derived'
OUT = ROOT / 'artifacts/nhanes15_notebook'
DERIVED.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.set_num_threads(1)
print('Project:', ROOT, '\nDữ liệu NHANES:', RAW)

## 2. Xác định dữ liệu và đặc trưng
### 2.1. Các tệp nguồn và 15 biến
`FILES` là danh sách cột cần từ từng tệp XPT; `FEATURES` là thứ tự đầu vào của mô hình. `DIQ010` là nhãn, không nằm trong `FEATURES`.

In [ ]:
FILES = {
    'DEMO': ['SEQN','RIDAGEYR','RIAGENDR','SDMVSTRA','SDMVPSU'],
    'DIQ': ['SEQN','DIQ010'], 'BMX': ['SEQN','BMXBMI','BMXWAIST','BMXWT','BMXHT'],
    'BPX': ['SEQN','BPXSY1','BPXDI1'], 'GLU': ['SEQN','LBXGLU'],
    'GHB': ['SEQN','LBXGH'], 'TCHOL': ['SEQN','LBXTC'], 'HDL': ['SEQN','LBDHDD'],
    'TRIGLY': ['SEQN','LBXTR'], 'BIOPRO': ['SEQN','LBXSATSI','LBXSCR']}
FEATURES = ['RIDAGEYR','RIAGENDR','BMXBMI','BMXWAIST','BPXSY1','BPXDI1',
            'LBXGLU','LBXGH','LBXTC','LBDHDD','LBXTR','LBXSATSI','LBXSCR','BMXWT','BMXHT']
assert len(FEATURES) == 15 and not set(FEATURES) & {'DIQ010', 'SEQN'}

### 2.2. Định nghĩa cách đọc, ghép và lọc
Ghép theo SEQN; chỉ giữ người từ 18 tuổi, có nhãn rõ ràng và đủ 15 biến. Thống kê dữ liệu thiếu và hash tệp nguồn.

In [ ]:
def read_cycle(year, suffix):
    frames, hashes = [], {}
    for name, columns in FILES.items():
        path = RAW / f'{name}_{suffix}.xpt'
        if not path.is_file():
            raise FileNotFoundError(f'Thiếu {path}; xem data/metadata/NHANES15.md để tải từ CDC.')
        raw = pd.read_sas(path, format='xport')
        missing = set(columns) - set(raw.columns)
        if missing or raw.SEQN.duplicated().any():
            raise ValueError(f'{path.name}: thiếu cột {missing} hoặc trùng SEQN')
        frames.append(raw[columns]); hashes[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
    # DEMO defines the population; left joins make missingness auditable.
    df = frames[0]
    for frame in frames[1:]:
        df = df.merge(frame, on='SEQN', how='left', validate='one_to_one')
    missingness = df[FEATURES].isna().sum().to_dict()
    eligible = df[(df.RIDAGEYR >= 18) & df.DIQ010.isin([1,2])].copy()
    for col in ['BPXSY1','BPXDI1']:
        eligible.loc[eligible[col] <= 0, col] = np.nan
    complete = eligible.dropna(subset=FEATURES + ['SDMVSTRA','SDMVPSU']).copy()
    complete['target'] = (complete.DIQ010 == 1).astype(int)
    complete['group'] = complete.SDMVSTRA.astype(str) + '_' + complete.SDMVPSU.astype(str)
    return complete, {'survey_year': year, 'demo_rows': len(df), 'eligible': len(eligible),
        'complete': len(complete), 'positive': int(complete.target.sum()),
        'missing_before_eligibility': missingness, 'sha256': hashes}

### 2.3. Đọc hai chu kỳ và đối chiếu
Đọc 2015–2016 và 2017–2018; in số mẫu, số nhãn dương và kiểm tra SEQN trùng trong các bản ghi được chọn.

In [ ]:
old, old_info = read_cycle(2015,'I')
new, new_info = read_cycle(2017,'J')
overlap = len(set(old.SEQN) & set(new.SEQN))
print('Chu kỳ 2015–2016:', {k:old_info[k] for k in ('demo_rows','eligible','complete','positive')})
print('Chu kỳ 2017–2018:', {k:new_info[k] for k in ('demo_rows','eligible','complete','positive')})
print('SEQN trùng giữa hai chu kỳ trong dữ liệu đã chọn:', overlap)
if overlap: raise ValueError('Có SEQN trùng giữa các chu kỳ; phải kiểm tra trước khi đánh giá.')

## 3. Chia tập và tiền xử lý
### 3.1. Tách train, validation và test
2015–2016 được chia theo cụm PSU thành train/validation; 2017–2018 giữ riêng làm test. Các kết quả mô tả mẫu, không tính trọng số khảo sát.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr_idx, va_idx = next(splitter.split(old, old.target, groups=old.group))
train, val, test = old.iloc[tr_idx].copy(), old.iloc[va_idx].copy(), new.copy()
assert set(train.group).isdisjoint(set(val.group))
for name, part in [('train',train),('validation',val),('test',test)]:
    counts = part.target.value_counts().to_dict()
    print(name, 'n=',len(part),'nhãn=',counts,'nhóm PSU=',part.group.nunique())
    if part.target.nunique() != 2: raise ValueError(f'{name} chỉ có một lớp; cần xem lại cách chia.')
    # Local derivative for reproducibility; retains public SEQN and group, never committed by .gitignore.
    part[['SEQN','SDMVSTRA','SDMVPSU', *FEATURES, 'target']].to_csv(DERIVED / f'{name}.csv', index=False)

### 3.2. Chuẩn hóa đặc trưng
Chỉ fit `StandardScaler` trên tập train; áp dụng cùng phép biến đổi lên validation và test để hạn chế rò rỉ.

In [ ]:
scaler = StandardScaler().fit(train[FEATURES])
Xtr, Xva, Xte = [scaler.transform(frame[FEATURES]).astype('float32') for frame in (train,val,test)]
ytr, yva, yte = [frame.target.to_numpy(dtype='float32') for frame in (train,val,test)]

## 4. Mô hình CNN 1D
### 4.1. Định nghĩa kiến trúc
CNN 1D gồm hai lớp tích chập, pooling và các lớp fully connected. Thứ tự 15 cột là giả định kỹ thuật, không có cấu trúc không gian tự nhiên.

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(1,16,3,padding=1), nn.ReLU(),
                                  nn.Conv1d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool1d(2))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(32*(n_features//2),32), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(32,1))
    def forward(self,x): return self.head(self.conv(x.unsqueeze(1))).squeeze(1)

### 4.2. Khởi tạo dữ liệu, loss và optimizer
Dùng `BCEWithLogitsLoss` có trọng số lớp và Adam. Đầu ra sigmoid chưa được hiệu chỉnh thành xác suất bệnh.

In [ ]:
model = CNN1D(len(FEATURES))
xt, yt = torch.tensor(Xtr), torch.tensor(ytr)
xv, yv = torch.tensor(Xva), torch.tensor(yva)
loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(xt,yt),batch_size=64,shuffle=True)
pos_weight = torch.tensor([(len(ytr)-sum(ytr))/sum(ytr)], dtype=torch.float32)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
best_loss, best_epoch, patience, best_state = float('inf'), 0, 0, None
history = []

### 4.3. Huấn luyện và dừng sớm
Theo dõi validation loss và lưu trọng số tại epoch tốt nhất; không sử dụng tập test ở bước này.

In [ ]:
for epoch in range(1,101):
    model.train()
    for xb,yb in loader:
        optimizer.zero_grad(); loss=criterion(model(xb),yb); loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad(): val_loss = float(criterion(model(xv),yv))
    history.append({'epoch':epoch, 'validation_loss':val_loss})
    if val_loss < best_loss - 1e-5:
        best_loss, best_epoch, patience = val_loss, epoch, 0
        best_state = {k:v.detach().clone() for k,v in model.state_dict().items()}
    else:
        patience += 1
        if patience >= 10: break
model.load_state_dict(best_state); model.eval()
print('Epoch validation tốt nhất:',best_epoch,'validation loss:',round(best_loss,4))

## 5. Mô hình Logistic Regression
Huấn luyện baseline cân bằng lớp trên cùng dữ liệu train đã chuẩn hóa để so sánh với CNN.

In [ ]:
baseline = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=SEED).fit(Xtr,ytr)

## 6. Đánh giá hai mô hình
### 6.1. Tạo điểm trên tập test
Tập test là chu kỳ 2017–2018, được giữ riêng trong toàn bộ bước huấn luyện.

In [ ]:
with torch.no_grad(): cnn_scores = torch.sigmoid(model(torch.tensor(Xte))).numpy()
log_scores = baseline.predict_proba(Xte)[:,1]

### 6.2. Tính và so sánh các chỉ số
Báo ROC-AUC, PR-AUC, accuracy, precision, recall, F1, Brier và ma trận nhầm lẫn tại ngưỡng 0,5 đã cố định trước. Không coi kết quả là kiểm định lâm sàng.

In [ ]:
def evaluate(y, scores):
    pred = scores >= 0.5
    return {'roc_auc':roc_auc_score(y,scores), 'pr_auc':average_precision_score(y,scores),
        'accuracy':accuracy_score(y,pred), 'precision':precision_score(y,pred,zero_division=0),
        'recall':recall_score(y,pred,zero_division=0), 'f1':f1_score(y,pred,zero_division=0),
        'brier':brier_score_loss(y,scores), 'confusion_matrix':confusion_matrix(y,pred,labels=[0,1]).tolist()}
results = {'cnn':evaluate(yte,cnn_scores),'logistic_regression':evaluate(yte,log_scores)}
print(json.dumps(results,ensure_ascii=False,indent=2))

## 7. Lưu dữ liệu và artifact
Lưu manifest (nguồn, hash, biến, seed, phiên bản, kết quả) cùng scaler và mô hình. Artifact này dành riêng cho notebook và không được website sử dụng.

In [ ]:
def hashes(path): return hashlib.sha256(path.read_bytes()).hexdigest()
manifest = {
    'dataset':'CDC NHANES USA 2015–2016 and 2017–2018; not WHO and not Vietnamese',
    'target':'DIQ010: 1 ever told diabetes, 2 no; excludes other values',
    'model_scope':'retrospective experimental survey-response classification; no diagnosis, screening or new-onset prediction',
    'source_urls':{f'{name}_{suffix}.xpt':f'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{name}_{suffix}.xpt' for year,suffix in [(2015,'I'),(2017,'J')] for name in FILES},
    'cycles':{'2015_2016':old_info,'2017_2018':new_info},
    'features_order':FEATURES, 'seed':SEED, 'split':'2015–2016 cluster grouped train/validation; 2017–2018 test',
    'n':{'train':len(train),'validation':len(val),'test':len(test)}, 'best_epoch':best_epoch,
    'metrics_unweighted':results, 'threshold':0.5, 'validation_history':history,
    'derived_sha256':{f'{s}.csv':hashes(DERIVED / f'{s}.csv') for s in ['train','validation','test']},
    'versions':{'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__,'sklearn':sklearn.__version__,'torch':torch.__version__},
    'executed_utc':datetime.now(timezone.utc).isoformat(),
    'limitations':['US NHANES, not Vietnamese','self-reported historical diagnosis, not incident disease',
                   'complete fasting cases and unweighted metrics','treatment and diagnostic testing affect predictors',
                   'no clinical validation or probability calibration','not for patient use']}
(OUT/'report.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
joblib.dump(scaler, OUT/'scaler.joblib'); joblib.dump(baseline, OUT/'logistic_regression.joblib')
torch.save({'state_dict':best_state,'features':FEATURES},OUT/'cnn.pt')
print('Đã lưu:',OUT,'và',DERIVED)

## 8. Giới hạn và hướng tiếp theo
- Nhãn là tự khai đã từng được thông báo mắc bệnh, không phải mắc mới hoặc chẩn đoán độc lập.
- Dữ liệu Mỹ, chọn trường hợp đầy đủ và không tính trọng số khảo sát; không suy rộng cho người Việt Nam.
- Glucose/HbA1c có thể liên quan tới chẩn đoán hoặc điều trị trước đó; điểm mô hình không phải xác suất mắc bệnh đã hiệu chỉnh.
- Khi có dữ liệu Việt Nam phù hợp, cần thiết kế bài toán, kiểm tra nhãn và huấn luyện notebook riêng; không ghi nguồn WHO cho mô hình hiện tại.